In [ ]:
# ── 1. Imports ────────────────────────────────────────────────────────────────
import os
import json
import time
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    f1_score,
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    roc_auc_score,
    matthews_corrcoef
)
from sklearn.preprocessing import label_binarize
import matplotlib

matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports loaded successfully!")

In [ ]:
# ── 2. Configuration ──────────────────────────────────────────────────────────
class Config:
    # Paths
    BASE_PATH = "/kaggle/input/datasets/mdkaifafrankhan/tal-grn-preprocessed-dataset/TAL_GRN_Preprocessed_dataset"
    TRAIN_CSV = "/kaggle/working/shuffled_csvs/train_shuffled.csv"
    VAL_CSV = "/kaggle/working/shuffled_csvs/val_shuffled.csv"
    TEST_CSV = "/kaggle/working/shuffled_csvs/test_shuffled.csv"

    # Hyperparameters
    IMAGE_SIZE = 224
    BATCH_SIZE = 64
    NUM_EPOCHS = 50
    LR = 1e-3
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 2
    SEED = 42
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Output directory for all method results
    RESULTS_BASE = Path("/kaggle/working/TAL_GRN_Imbalance_Methods")
    RESULTS_BASE.mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(Config.SEED)
np.random.seed(Config.SEED)
torch.manual_seed(Config.SEED)
torch.cuda.manual_seed_all(Config.SEED)

print(f"Device: {Config.DEVICE}")
print(f"Results directory: {Config.RESULTS_BASE}")

In [ ]:
# ── 3. Dataset with Transformations ──────────────────────────────────────────
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_transform = T.Compose([
    T.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(degrees=10),
    T.ColorJitter(brightness=0.2, contrast=0.1, saturation=0.1),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = T.Compose([
    T.Resize((Config.IMAGE_SIZE, Config.IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

class MedicalImageDataset(Dataset):
    def __init__(self, csv_path: str, transform=None):
        self.df = pd.read_csv(csv_path)
        self.transform = transform

        assert 'image_path' in self.df.columns, "CSV must have 'image_path' column"
        assert 'label_id' in self.df.columns, "CSV must have 'label_id' column"

        self.paths = self.df['image_path'].tolist()
        self.labels = self.df['label_id'].tolist()

        if 'label' in self.df.columns:
            self.class_names = self.df['label'].tolist()
        else:
            self.class_names = [str(l) for l in self.labels]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_path = self.paths[idx]
        label = int(self.labels[idx])

        try:
            img = Image.open(img_path).convert('RGB')
        except Exception:
            img = Image.fromarray(
                np.zeros((Config.IMAGE_SIZE, Config.IMAGE_SIZE, 3), dtype=np.uint8)
            )

        if self.transform:
            img = self.transform(img)

        return img, label

def get_class_distribution():
    """Get class distribution from training set"""
    train_df = pd.read_csv(Config.TRAIN_CSV)
    class_counts = train_df['label_id'].value_counts().sort_index().tolist()
    return class_counts, len(class_counts)

def build_loaders():
    train_ds = MedicalImageDataset(Config.TRAIN_CSV, transform=train_transform)
    val_ds = MedicalImageDataset(Config.VAL_CSV, transform=eval_transform)
    test_ds = MedicalImageDataset(Config.TEST_CSV, transform=eval_transform)

    num_classes = len(set(train_ds.labels))
    print(f"\nClasses: {num_classes}")
    print(f"Train samples: {len(train_ds)}")
    print(f"Val samples: {len(val_ds)}")
    print(f"Test samples: {len(test_ds)}")

    train_loader = DataLoader(
        train_ds, batch_size=Config.BATCH_SIZE, shuffle=True,
        num_workers=Config.NUM_WORKERS, pin_memory=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=True
    )
    test_loader = DataLoader(
        test_ds, batch_size=Config.BATCH_SIZE, shuffle=False,
        num_workers=Config.NUM_WORKERS, pin_memory=True
    )

    return train_loader, val_loader, test_loader, num_classes, train_ds

In [ ]:
# ── 4. Loss Functions for Imbalance Mitigation ──────────────────────────────

class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha  # Class weights tensor

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal_loss = ((1 - pt) ** self.gamma * ce).mean()
        return focal_loss

def get_class_balanced_weights(class_counts, beta=0.9999):
    """Compute Class-Balanced Loss weights"""
    effective_num = 1.0 - np.power(beta, class_counts)
    weights = (1.0 - beta) / np.array(effective_num)
    weights = weights / weights.sum() * len(class_counts)
    return torch.tensor(weights, dtype=torch.float)

class LDAMLoss(nn.Module):
    """LDAM (Label-Distribution-Aware Margin) Loss with DRW"""
    def __init__(self, cls_num_list, max_m=0.5, s=30):
        super().__init__()
        m_list = 1.0 / np.sqrt(np.sqrt(cls_num_list))
        m_list = m_list * (max_m / np.max(m_list))
        self.m_list = torch.FloatTensor(m_list).to(Config.DEVICE)
        self.s = s
        self.cls_num_list = cls_num_list

    def forward(self, x, target, weight=None):
        index = torch.zeros_like(x, dtype=torch.bool)
        index.scatter_(1, target.unsqueeze(1), True)
        x_m = x - self.m_list
        output = torch.where(index, x_m, x)
        return F.cross_entropy(self.s * output, target, weight=weight)

def logit_adjustment_loss(logits, targets, cls_num_list, tau=1.0):
    """Logit Adjustment Loss"""
    log_prior = torch.log(
        torch.tensor(cls_num_list, dtype=torch.float) / sum(cls_num_list)
    ).to(Config.DEVICE)
    adjusted_logits = logits + tau * log_prior.unsqueeze(0)
    return F.cross_entropy(adjusted_logits, targets)

In [ ]:
# ── 5. Model Builder ──────────────────────────────────────────────────────────
def build_model(num_classes: int) -> nn.Module:
    """Frozen ResNet-50 backbone + trainable FC head"""
    model = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

    # Freeze backbone
    for param in model.parameters():
        param.requires_grad = False

    # Replace FC layer
    in_features = model.fc.in_features
    model.fc = nn.Linear(in_features, num_classes)

    return model.to(Config.DEVICE)

def count_parameters(model):
    """Count trainable and total parameters"""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

In [ ]:
# ── 6. Training and Evaluation Functions ──────────────────────────────────────
def compute_metrics(all_labels, all_preds, all_probs, num_classes):
    """Compute comprehensive metrics"""
    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)

    # Basic metrics
    accuracy = accuracy_score(all_labels, all_preds)
    balanced_acc = balanced_accuracy_score(all_labels, all_preds)

    # Precision, Recall, F1 (macro)
    precision_macro = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    recall_macro = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_macro = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    f1_weighted = f1_score(all_labels, all_preds, average='weighted', zero_division=0)

    # MCC
    mcc = matthews_corrcoef(all_labels, all_preds)

    # ROC-AUC (if we have probabilities)
    auc_ovr, auc_ovo = 0, 0
    if all_probs is not None and len(all_probs) > 0:
        try:
            # Binarize labels for multi-class ROC
            all_labels_bin = label_binarize(all_labels, classes=range(num_classes))
            auc_ovr = roc_auc_score(all_labels_bin, all_probs, average='macro', multi_class='ovr')
            auc_ovo = roc_auc_score(all_labels_bin, all_probs, average='macro', multi_class='ovo')
        except Exception as e:
            print(f"Warning: Could not compute AUC - {e}")

    return {
        'accuracy': accuracy,
        'balanced_accuracy': balanced_acc,
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'f1_weighted': f1_weighted,
        'mcc': mcc,
        'auc_ovr': auc_ovr,
        'auc_ovo': auc_ovo
    }

def run_epoch(model, loader, criterion, optimizer=None, is_train=False):
    if is_train:
        model.train()
    else:
        model.eval()

    total_loss = 0.0
    all_preds = []
    all_labels = []
    all_probs = []

    ctx = torch.enable_grad() if is_train else torch.no_grad()

    with ctx:
        for images, labels in loader:
            images = images.to(Config.DEVICE, non_blocking=True)
            labels = labels.to(Config.DEVICE, non_blocking=True)

            logits = model(images)

            # Handle different loss function signatures
            if isinstance(criterion, LDAMLoss):
                # LDAM needs special handling
                loss = criterion(logits, labels)
            elif 'logit_adjustment' in str(criterion.__name__):
                loss = criterion(logits, labels)
            else:
                loss = criterion(logits, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * images.size(0)
            probs = F.softmax(logits, dim=1)
            preds = logits.argmax(dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    n = len(all_labels)
    avg_loss = total_loss / n
    metrics = compute_metrics(all_labels, all_preds, all_probs, logits.shape[1])

    return avg_loss, metrics, all_preds, all_labels

def train_model(model, train_loader, val_loader, criterion, method_name, class_counts):
    """Train model with given criterion"""
    optimizer = optim.AdamW(model.fc.parameters(), lr=Config.LR, weight_decay=Config.WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=Config.NUM_EPOCHS)

    best_val_f1 = -1.0
    best_epoch = -1
    history = []

    method_dir = Config.RESULTS_BASE / method_name
    method_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*70}")
    print(f"Training with {method_name}")
    print(f"{'='*70}")
    print(f"{'Ep':>4}  {'Tr-Loss':>8}  {'Tr-Acc':>7}  {'Tr-F1':>6}  {'Va-Loss':>8}  {'Va-Acc':>7}  {'Va-F1':>6}  {'Time':>6}")
    print("-" * 70)

    for epoch in range(1, Config.NUM_EPOCHS + 1):
        t0 = time.time()

        # Update weights for DRW if using LDAM
        if isinstance(criterion, LDAMLoss) and epoch > Config.NUM_EPOCHS * 0.8:
            cb_weights = get_class_balanced_weights(class_counts).to(Config.DEVICE)
            # For LDAM, we need to pass weights differently
            criterion.weight = cb_weights

        # Train
        tr_loss, tr_metrics, _, _ = run_epoch(model, train_loader, criterion, optimizer, is_train=True)

        # Validate
        va_loss, va_metrics, _, _ = run_epoch(model, val_loader, criterion, is_train=False)

        scheduler.step()
        elapsed = time.time() - t0

        print(f"{epoch:>4}  {tr_loss:>8.4f}  {tr_metrics['accuracy']:>7.4f}  {tr_metrics['f1_macro']:>6.4f}  "
              f"{va_loss:>8.4f}  {va_metrics['accuracy']:>7.4f}  {va_metrics['f1_macro']:>6.4f}  {elapsed:>5.1f}s")

        history.append({
            'epoch': epoch,
            'train_loss': tr_loss,
            'train_acc': tr_metrics['accuracy'],
            'train_f1': tr_metrics['f1_macro'],
            'val_loss': va_loss,
            'val_acc': va_metrics['accuracy'],
            'val_f1': va_metrics['f1_macro']
        })

        if va_metrics['f1_macro'] > best_val_f1:
            best_val_f1 = va_metrics['f1_macro']
            best_epoch = epoch
            torch.save(model.state_dict(), method_dir / 'best_model.pth')

    return best_epoch, best_val_f1, history

In [ ]:
# ── 7. Evaluation with Comprehensive Metrics ─────────────────────────────────
def evaluate_model(model, test_loader, method_name, num_classes, class_names):
    """Evaluate model and generate full report"""
    model.load_state_dict(torch.load(
        Config.RESULTS_BASE / method_name / 'best_model.pth',
        map_location=Config.DEVICE
    ))

    criterion = nn.CrossEntropyLoss()
    _, metrics, all_preds, all_labels = run_epoch(model, test_loader, criterion, is_train=False)

    # Generate classification report
    report = classification_report(
        all_labels, all_preds,
        target_names=class_names,
        zero_division=0,
        digits=4
    )

    # Prepare results dictionary
    results = {
        'method': method_name,
        'test_accuracy': metrics['accuracy'],
        'test_balanced_accuracy': metrics['balanced_accuracy'],
        'test_precision_macro': metrics['precision_macro'],
        'test_recall_macro': metrics['recall_macro'],
        'test_f1_macro': metrics['f1_macro'],
        'test_f1_weighted': metrics['f1_weighted'],
        'test_mcc': metrics['mcc'],
        'test_auc_ovr': metrics['auc_ovr'],
        'test_auc_ovo': metrics['auc_ovo']
    }

    # Save results
    method_dir = Config.RESULTS_BASE / method_name
    with open(method_dir / 'test_metrics.json', 'w') as f:
        json.dump(results, f, indent=4)

    with open(method_dir / 'classification_report.txt', 'w') as f:
        f.write(report)

    # Plot confusion matrix
    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    fig, ax = plt.subplots(figsize=(14, 12))
    sns.heatmap(cm, ax=ax, xticklabels=class_names, yticklabels=class_names,
                cmap='Blues', fmt='.2f', linewidths=0.3)
    ax.set_title(f'{method_name} - Normalized Confusion Matrix', fontsize=14)
    ax.set_xlabel('Predicted', fontsize=12)
    ax.set_ylabel('True', fontsize=12)
    plt.xticks(rotation=90, fontsize=8)
    plt.yticks(rotation=0, fontsize=8)
    plt.tight_layout()
    fig.savefig(method_dir / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    plt.close()

    return results, report

In [ ]:
# ── 8. Main Training Pipeline for All Methods ───────────────────────────────
def main():
    print("=" * 70)
    print("TAL-GRN — Class Imbalance Mitigation Experiment")
    print("Comparing 5 different methods")
    print("=" * 70)

    # Load data and get class distribution
    train_loader, val_loader, test_loader, num_classes, train_ds = build_loaders()
    class_counts, num_classes = get_class_distribution()

    print(f"\nClass distribution (first 10 classes):")
    for i, count in enumerate(class_counts[:10]):
        print(f"  Class {i}: {count} samples")
    print(f"  ... ({num_classes} total classes)")

    # Get class names
    test_df = pd.read_csv(Config.TEST_CSV)
    if 'label' in test_df.columns and 'label_id' in test_df.columns:
        idx_to_name = test_df.groupby('label_id')['label'].first().sort_index().to_dict()
        class_names = [idx_to_name.get(i, str(i)) for i in range(num_classes)]
    else:
        class_names = [str(i) for i in range(num_classes)]

    # Prepare weights for different methods
    train_labels = train_ds.labels
    label_counts = Counter(train_labels)
    class_weights = torch.tensor([
        len(train_labels) / (num_classes * label_counts[i])
        for i in range(num_classes)
    ], dtype=torch.float).to(Config.DEVICE)

    cb_weights = get_class_balanced_weights(class_counts).to(Config.DEVICE)

    # Define methods to test
    methods = {
        '1_Weighted_CE': {
            'criterion': nn.CrossEntropyLoss(weight=class_weights)
        },
        '2_Focal_Loss': {
            'criterion': FocalLoss(gamma=2.0, alpha=class_weights)
        },
        '3_CB_Loss': {
            'criterion': nn.CrossEntropyLoss(weight=cb_weights)
        },
        '4_LDAM_DRW': {
            'criterion': LDAMLoss(cls_num_list=class_counts, max_m=0.5, s=30)
        },
        '5_Logit_Adjustment': {
            'criterion': lambda logits, targets: logit_adjustment_loss(
                logits, targets, class_counts, tau=1.0
            )
        }
    }

    all_results = {}

    # Train and evaluate each method
    for method_name, method_config in methods.items():
        print(f"\n{'#'*70}")
        print(f"# Method: {method_name}")
        print(f"{'#'*70}")

        # Build fresh model for each method
        model = build_model(num_classes)
        trainable, total = count_parameters(model)
        print(f"\nModel parameters - Trainable: {trainable:,} | Total: {total:,}")

        # Train
        best_epoch, best_val_f1, history = train_model(
            model, train_loader, val_loader, method_config['criterion'],
            method_name, class_counts
        )

        print(f"\n✓ Best val F1: {best_val_f1:.4f} at epoch {best_epoch}")

        # Evaluate
        results, report = evaluate_model(
            model, test_loader, method_name, num_classes, class_names
        )

        all_results[method_name] = results

        # Print test results
        print(f"\n{'='*60}")
        print(f"TEST RESULTS - {method_name}")
        print(f"{'='*60}")
        print(f"  Loss: Cross-Entropy (weighted variant)")
        print(f"  Accuracy: {results['test_accuracy']:.4f}")
        print(f"  Balanced Accuracy: {results['test_balanced_accuracy']:.4f}")
        print(f"  Precision (Macro): {results['test_precision_macro']:.4f}")
        print(f"  Recall/Sensitivity (Macro): {results['test_recall_macro']:.4f}")
        print(f"  F1-Score (Macro): {results['test_f1_macro']:.4f}")
        print(f"  F1-Score (Weighted): {results['test_f1_weighted']:.4f}")
        print(f"  ROC-AUC (Macro OvR): {results['test_auc_ovr']:.4f}")
        print(f"  ROC-AUC (Macro OvO): {results['test_auc_ovo']:.4f}")
        print(f"  MCC: {results['test_mcc']:.4f}")
        print(f"{'='*60}")

        # Print classification report snippet
        print("\nClassification Report (first 10 classes):")
        report_lines = report.split('\n')
        for i, line in enumerate(report_lines[:min(15, len(report_lines))]):
            print(f"  {line}")
        print("  ...")

        # Clean up GPU memory
        del model
        torch.cuda.empty_cache()

    # Comparison summary
    print(f"\n{'='*80}")
    print("FINAL COMPARISON - ALL METHODS")
    print(f"{'='*80}")

    comparison_df = pd.DataFrame(all_results).T
    print("\n", comparison_df.round(4))

    # Save comparison
    comparison_df.to_csv(Config.RESULTS_BASE / 'methods_comparison.csv')

    # Plot comparison bar chart
    metrics_to_plot = ['test_accuracy', 'test_f1_macro', 'test_f1_weighted', 'test_mcc']
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    axes = axes.flatten()

    for idx, metric in enumerate(metrics_to_plot):
        methods_names = list(all_results.keys())
        values = [all_results[m][metric] for m in methods_names]

        bars = axes[idx].bar(methods_names, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'])
        axes[idx].set_title(metric.replace('test_', '').replace('_', ' ').title(), fontsize=12)
        axes[idx].set_ylabel('Score')
        axes[idx].set_ylim([0, 1])
        axes[idx].tick_params(axis='x', rotation=45, labelsize=9)

        # Add value labels on bars
        for bar, val in zip(bars, values):
            axes[idx].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                          f'{val:.3f}', ha='center', va='bottom', fontsize=9)

    plt.suptitle('Comparison of Class Imbalance Mitigation Methods', fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig(Config.RESULTS_BASE / 'methods_comparison.png', dpi=150, bbox_inches='tight')
    plt.close()

    # Print best method
    best_method = max(all_results.keys(), key=lambda x: all_results[x]['test_f1_macro'])
    print(f"\n{'='*80}")
    print(f"🏆 BEST METHOD: {best_method}")
    print(f"   Test Macro F1: {all_results[best_method]['test_f1_macro']:.4f}")
    print(f"   Test Accuracy: {all_results[best_method]['test_accuracy']:.4f}")
    print(f"   Test MCC: {all_results[best_method]['test_mcc']:.4f}")
    print(f"{'='*80}")

    print(f"\n✅ All results saved to: {Config.RESULTS_BASE}")
    print("   - Individual method folders with best_model.pth, confusion matrices, etc.")
    print("   - methods_comparison.csv - Overall comparison")
    print("   - methods_comparison.png - Visualization")

if __name__ == '__main__':
    main()